# VaR 风险测算 - 定增项目

## 分析目标
使用多种方法计算定增项目的风险价值 (VaR)，包括：
- 历史模拟法
- 方差-协方差法（参数法）
- 蒙特卡洛法
- CVaR（条件风险价值）
- 最大回撤分析

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from utils.analysis_tools import PrivatePlacementRiskAnalyzer

# 配置中文字体
from utils.font_manager import get_font_prop
font_prop = get_font_prop()

%matplotlib inline
sns.set_style('whitegrid')

print('✅ VaR风险测算模块加载成功')

## 1. 参数设置

In [ ]:
# 定增项目参数
PROJECT_PARAMS = {
    'issue_price': 20.0,
    'issue_shares': 5000000,
    'lockup_period': 12,
    'current_price': 18.5,
    'risk_free_rate': 0.03
}

# 风险参数
RISK_PARAMS = {
    'volatility': 0.35,
    'confidence_levels': [0.90, 0.95, 0.99]
}

# 创建分析器
analyzer = PrivatePlacementRiskAnalyzer(**PROJECT_PARAMS)

print('=== 项目参数 ===')
print(f"发行价格: {PROJECT_PARAMS['issue_price']} 元/股")
print(f"当前价格: {PROJECT_PARAMS['current_price']} 元/股")
print(f"锁定期: {PROJECT_PARAMS['lockup_period']} 个月")
print(f"波动率: {RISK_PARAMS['volatility']*100:.0f}%")

## 2. 历史模拟法 VaR

In [ ]:
# 生成历史收益率样本（模拟）
np.random.seed(42)
n_samples = 1000

# 模拟历史收益率（假设正态分布）
historical_returns = np.random.normal(
    loc=0.08,  # 平均年化收益率 8%
    scale=RISK_PARAMS['volatility'],
    size=n_samples
)

# 计算定增收益率（锁定期收益）
lockup_return = (1 + historical_returns) ** (PROJECT_PARAMS['lockup_period'] / 12) - 1

# 计算VaR
historical_var = {}
for cl in RISK_PARAMS['confidence_levels']:
    var = abs(np.percentile(lockup_return, (1 - cl) * 100))
    historical_var[cl] = var

print('=== 历史模拟法 VaR ===')
for cl, var in historical_var.items():
    print(f"{int(cl*100)}% VaR: {var*100:.2f}%")

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. 收益率分布
ax1.hist(lockup_return * 100, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
for cl, var in historical_var.items():
    ax1.axvline(x=-var*100, color='red', linestyle='--', 
               label=f'{int(cl*100)}% VaR', linewidth=2)
ax1.axvline(x=0, color='green', linestyle='-', linewidth=1, label='盈亏平衡')
ax1.set_xlabel('收益率 (%)', fontsize=12, fontproperties=font_prop)
ax1.set_ylabel('频数', fontsize=12, fontproperties=font_prop)
ax1.set_title('历史模拟法 - 收益率分布', fontsize=14, fontweight='bold', fontproperties=font_prop)
ax1.legend(prop=font_prop)
ax1.grid(True, alpha=0.3)

# 2. VaR柱状图
cl_labels = [f"{int(cl*100)}%" for cl in historical_var.keys()]
var_values = [var * 100 for var in historical_var.values()]
bars = ax2.bar(cl_labels, var_values, color='coral', alpha=0.7)
ax2.set_xlabel('置信水平', fontsize=12, fontproperties=font_prop)
ax2.set_ylabel('VaR (%)', fontsize=12, fontproperties=font_prop)
ax2.set_title('历史模拟法 - VaR对比', fontsize=14, fontweight='bold', fontproperties=font_prop)
ax2.grid(True, axis='y', alpha=0.3)
for bar, value in zip(bars, var_values):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{value:.2f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 3. 参数法 VaR（方差-协方差法）

In [ ]:
# 参数法假设收益率服从正态分布
drift = 0.08  # 年化漂移率
volatility = RISK_PARAMS['volatility']

# 计算锁定期收益率参数
lockup_drift = drift * (PROJECT_PARAMS['lockup_period'] / 12)
lockup_vol = volatility * np.sqrt(PROJECT_PARAMS['lockup_period'] / 12)

# 计算VaR（正态分布）
parametric_var = {}
for cl in RISK_PARAMS['confidence_levels']:
    # 使用标准正态分布的分位数
    z_score = stats.norm.ppf(1 - cl)
    var = abs(lockup_drift + z_score * lockup_vol)
    parametric_var[cl] = var

print('=== 参数法 VaR ===')
print(f"锁定期漂移率: {lockup_drift*100:.2f}%")
print(f"锁定期波动率: {lockup_vol*100:.2f}%")
print()
for cl, var in parametric_var.items():
    print(f"{int(cl*100)}% VaR: {var*100:.2f}%")

## 4. 蒙特卡洛法 VaR

In [ ]:
# 运行蒙特卡洛模拟
print("运行蒙特卡洛模拟...")

n_simulations = 10000
sim_results = analyzer.monte_carlo_simulation(
    n_simulations=n_simulations,
    volatility=RISK_PARAMS['volatility'],
    drift=0.08,
    seed=42
)

# 计算锁定期末的收益率
lockup_days = PROJECT_PARAMS['lockup_period'] * 30
final_prices = sim_results.iloc[:, lockup_days].values
mc_returns = (final_prices - PROJECT_PARAMS['issue_price']) / PROJECT_PARAMS['issue_price']

# 计算VaR
mc_var = {}
for cl in RISK_PARAMS['confidence_levels']:
    var = abs(np.percentile(mc_returns, (1 - cl) * 100))
    mc_var[cl] = var

print(f"✅ 模拟完成！共 {n_simulations:,} 次模拟")
print('\n=== 蒙特卡洛法 VaR ===')
for cl, var in mc_var.items():
    print(f"{int(cl*100)}% VaR: {var*100:.2f}%")

## 5. CVaR（条件风险价值）

In [ ]:
# CVaR 是超过VaR的平均损失
def calculate_cvar(returns, var, confidence_level):
    """计算条件风险价值（Expected Shortfall）"""
    # 找出所有低于VaR的收益率
    tail_losses = returns[returns <= -var]
    if len(tail_losses) > 0:
        cvar = abs(tail_losses.mean())
    else:
        cvar = var
    return cvar

# 使用蒙特卡洛结果计算CVaR
mc_cvar = {}
for cl in RISK_PARAMS['confidence_levels']:
    var = mc_var[cl]
    cvar = calculate_cvar(mc_returns, var, cl)
    mc_cvar[cl] = cvar

print('=== CVaR（条件风险价值）===')
print('基于蒙特卡洛模拟结果')
print()
for cl in RISK_PARAMS['confidence_levels']:
    print(f"{int(cl*100)}% 置信水平:")
    print(f"  VaR:  {mc_var[cl]*100:.2f}%")
    print(f"  CVaR: {mc_cvar[cl]*100:.2f}%")
    print()

## 6. 最大回撤分析

In [ ]:
# 计算最大回撤
max_dd = analyzer.calculate_max_drawdown(
    current_price=PROJECT_PARAMS['current_price'],
    volatility=RISK_PARAMS['volatility'],
    confidence_level=0.95
)

# 模拟多条路径计算回撤分布
n_paths = 1000
drawdowns = []

for i in range(n_paths):
    path = sim_results.iloc[i, :lockup_days+1].values
    
    # 计算累计最大值
    cummax = np.maximum.accumulate(path)
    
    # 计算回撤
    dd = (path - cummax) / cummax
    
    # 找到最大回撤
    max_dd_path = abs(dd.min())
    drawdowns.append(max_dd_path)

print('=== 最大回撤分析 ===')
print(f"理论95%置信最大回撤: {max_dd*100:.2f}%")
print(f"\n模拟回撤统计 ({n_paths}条路径):")
print(f"  平均最大回撤: {np.mean(drawdowns)*100:.2f}%")
print(f"  中位数最大回撤: {np.median(drawdowns)*100:.2f}%")
print(f"  95%分位回撤: {np.percentile(drawdowns, 95)*100:.2f}%")

## 7. VaR 风险测算结论

In [ ]:
print('\n' + '='*60)
print('VaR风险测算结论')
print('='*60)

print(f"\n📊 项目参数:")
print(f"   发行价格: {PROJECT_PARAMS['issue_price']} 元/股")
print(f"   当前价格: {PROJECT_PARAMS['current_price']} 元/股")
print(f"   锁定期: {PROJECT_PARAMS['lockup_period']} 个月")
print(f"   波动率: {RISK_PARAMS['volatility']*100:.0f}%")

print(f"\n⚠️ VaR测算结果（蒙特卡洛法）:")
for cl in RISK_PARAMS['confidence_levels']:
    print(f"   {int(cl*100)}% VaR: {mc_var[cl]*100:.2f}%")
    print(f"   {int(cl*100)}% CVaR: {mc_cvar[cl]*100:.2f}%")
    print()

print(f"\n📉 最大回撤分析:")
print(f"   平均最大回撤: {np.mean(drawdowns)*100:.2f}%")
print(f"   95%分位回撤: {np.percentile(drawdowns, 95)*100:.2f}%")

# 计算损失金额
investment = PROJECT_PARAMS['issue_price'] * PROJECT_PARAMS['issue_shares']
loss_95_var = investment * mc_var[0.95]
loss_95_cvar = investment * mc_cvar[0.95]

print(f"\n💰 潜在损失金额:")
print(f"   投资总额: {investment/10000:.0f} 万元")
print(f"   95% VaR潜在损失: {loss_95_var/10000:.2f} 万元")
print(f"   95% CVaR潜在损失: {loss_95_cvar/10000:.2f} 万元")